In [ ]:
import seaborn as sns
import pandas as pd
import matplotlib.pyplot as plt

import sys
sys.path.append('../')
from utils import * #only needed for xgboost
from friedman1 import *

import warnings
warnings.filterwarnings("ignore") #just to supress warnings

In [ ]:
#Function that returns best params for each pair of (d, target_instance) 
# and their average val rmse, rmse, mae, given a set of config columns (depending on method)
# This funcion can take a dataframe or a path to a csv as input.
# the best settings are chosen using the lowest val rmse
def topk_per_d_per_method(data, config_cols, k=5):
    """
    Return up to the top-k configs per (target_instances, d), ranked by avg_rmse.
    Averages are computed across seeds.
    """
    df = pd.read_csv(data, index_col=[0]) if isinstance(data, str) else data.copy()
    df = df.sort_values(by=['seed'] + config_cols)

    # aggregate over seeds
    agg = (df.groupby(config_cols, as_index=False)
             .agg(avg_rmse=('rmse', 'mean'),
                  #avg_mae=('val_mae', 'mean'),
                  n_seeds=('seed', 'nunique'),
                  d=('d', 'first')))

    # merge back on config_cols only (minimal fix)
    agg = agg.merge(df, on=config_cols, how='left')

    # sort globally by the fields that determine the ranking
    agg = agg.sort_values(by=['d', 'avg_rmse'], ascending=True)

    # select top-k per (target_instances, d)
    topk_df = agg.groupby(['d'], group_keys=False).head(k)

    # final compact output
    topk_d = topk_df[['d', 'avg_rmse', 'rmse', 'mae']]

    return topk_d, agg




In [ ]:
size = 200
#TTB had parallell runs, so here they are merged
data_ttb = pd.read_csv(f'results_{size}/ttb_LS.csv')
data_ttb_ = pd.read_csv(f'results_{size}/ttb_LS_936937.csv')
data_ttb__ = pd.read_csv(f'results_{size}/ttb_LS_934935.csv')
data_ttb___ = pd.read_csv(f'results_{size}/ttb_LS_938939.csv')
data_ttb = pd.concat([data_ttb, data_ttb_])
data_ttb = pd.concat([data_ttb, data_ttb__])
data_ttb = pd.concat([data_ttb, data_ttb___])

data_xgboost = pd.read_csv(f'results_{size}/xgb.csv')

data_xgboost_warmstart = pd.read_csv(f'results_{size}/xgb_warmstart.csv')

data_xgboost_pooled = pd.read_csv(f'results_{size}/xgb_naive.csv')

data_trada = pd.read_csv('results_200/trada.csv')
data_ttb = data_ttb.drop_duplicates(subset = ['seed', 'd', 'v', 'target_tree_size', 'source_tree_size', 'm_0', 'k']) #drop duplicates for TTB
len(data_ttb)

In [ ]:

plt.figure(figsize = (7,7))

import matplotlib as mpl

best_ttb, best_ttb_params = topk_per_d_per_method(
    data_ttb,
    ['d', 'v', 'source_tree_size', 'target_tree_size', 'k', 'm_0'], k=10
)
best_ttb['Method'] = 'TransferTreeBoost'

best_xgboost, best_xgboost_params = topk_per_d_per_method(
    data_xgboost,
    ['d', 'v', 'target_tree_size'], k=10
)
best_xgboost['Method'] = 'XGBoost'

best_xgboost_warmstart, best_xgboost_warmstart_params = topk_per_d_per_method(
    data_xgboost_warmstart,
    ['d', 'v', 'target_tree_size'], k=10
)
best_xgboost_warmstart['Method'] = 'XGBoost Warmstart'

best_xgboost_pooled, best_xgboost_pooled_params = topk_per_d_per_method(
    data_xgboost_pooled,
    ['d', 'v', 'target_tree_size'], k=10
)
best_xgboost_pooled['Method'] = 'Pooled XGBoost'

best_trada, best_trada_params = topk_per_d_per_method(
    data_trada,
    ['d', 'lr', 'n_estimators', 'tree_size'], k=10
)
best_trada['Method'] = 'TrAdaBoost.R2'


# create new df for viz
df = pd.concat([best_ttb, best_xgboost])
df = pd.concat([df, best_xgboost_warmstart])
df = pd.concat([df, best_xgboost_pooled])
df = pd.concat([df, best_trada]) #Comment line for figure without trada
plt.figure(figsize = (20,10))
sns.lineplot(data=df, x='d', y='rmse', hue='Method', style = 'Method', markers = {'TransferTreeBoost':'o',
                                                                                  'XGBoost': 's', 'XGBoost Warmstart': '^',
                                                                                   'Pooled XGBoost': 'P', 'TrAdaBoost.R2': 'p'
                                                                                    },
                                                                                    markersize = 12)
plt.legend(title = None, fontsize = 20, title_fontsize = 20)
plt.xticks([3, 6, 9, 12, 15], size = 12)  # positions of ticks
plt.yticks(size = 12)  # positions of ticks
plt.xlabel('$d$', fontsize = 18)
plt.ylabel('RMSE', fontsize = 18)
plt.savefig('vizes/final_with_trada.png', bbox_inches = 'tight', pad_inches = 0.1, dpi = 500)
